# **Getting the data by upload it manually since it's small**
www.kaggle.com/datasets/sinjoysaha/sales-analysis-dataset


In [1]:
# Getting the Data

import modin.pandas as pd
import os
import glob

os.environ["MODIN_ENGINE"] = "dask"
# getting the folder path
folder = r"C:\Users\farou\Downloads\Sales-Analysis-Dataset"

# Reading every file in the folder
files = glob.glob(os.path.join(folder,"*.csv"))
data_list = [pd.read_csv(file) for file in files]
# combine the files in one dataframe
df = pd.concat(data_list,ignore_index=True)
# Checking the data
print(df["Order Date"].head())

# Transforming the dates in the data into pandas date-type
df["Order Date"] = pd.to_datetime(df["Order Date"],format="%m/%d/%y %H:%M",errors="coerce")
df = df.dropna(subset=["Order Date"])
df["Date"] = df["Order Date"].dt.date
df["Month"] = df["Order Date"].dt.month

# Sorting the data by the month e.g: 1-2-3-4
df = df.sort_values(by="Month",ascending=True)
# we transform any column-number has a string in it e.g:"5" into a int or float 
cols_to_convert = ["Price Each","Quantity Ordered"]
for col in cols_to_convert:
    df[col] = pd.to_numeric(df[col],errors="coerce")

df.info()


0    04/19/19 08:46
1               NaN
2    04/07/19 22:30
3    04/12/19 14:38
4    04/12/19 14:38
Name: Order Date, dtype: object
<class 'modin.pandas.dataframe.DataFrame'>
Index: 185950 entries, 31957 to 30394
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Order ID          185950 non-null  object        
 1   Product           185950 non-null  object        
 2   Quantity Ordered  185950 non-null  int64         
 3   Price Each        185950 non-null  float64       
 4   Order Date        185950 non-null  datetime64[ns]
 5   Purchase Address  185950 non-null  object        
 6   Date              185950 non-null  object        
 7   Month             185950 non-null  int32         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(1), object(4)
memory usage: 12.1+ MB


# **Prepare The Data** **ℾ**

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Confirm that the data has been sorted by the month correctly
print(df.head())
# Transform the rest into pandas date type
df["Year"] = df["Order Date"].dt.year
df["Day"] = df["Order Date"].dt.day
df["Weekend"] = df["Order Date"].dt.dayofweek
df["is_weekend"] = df["Weekend"].apply(lambda x: 1 if x in [5,6] else 0)
delete = ["Order ID","Order Date","Purchase Address"]
df = df.drop(columns=delete)
df = df.dropna()
# changine the names of the products
conditions = [
    df['Price Each'].isin([2.99, 3.84]),                            
    df['Price Each'].isin([11.95, 14.95, 11.99]),                   
    df['Price Each'].isin([99.99, 109.99]),                         
    df['Price Each'].isin([149.99, 150.00]),                        
    df['Price Each'].isin([300.00, 379.99, 389.99, 400.00]),        
    df['Price Each'].isin([600.00, 700.00]),                        
    df['Price Each'].isin([999.99, 1700.00])                        
]
choices = [
    "Phone accessories", 
    'Phone cables', 
    'Bluetooth Headphones', 
    'Smartwatches', 
    'Washing machines', 
    'Air conditioners', 
    'Refrigerators'
]

df["Product"] = np.select(conditions,choices,default="Other")
# Spliting the data by applying Stratified Sampling
max_val = df["Quantity Ordered"].max()
min_val = df["Quantity Ordered"].min()
df["Quantity Ordered_cat"] = pd.cut(df["Quantity Ordered"],bins=[min_val,3,5,7,9,max_val],labels=False,duplicates="drop",include_lowest=True)
df_new = df.dropna(subset=["Quantity Ordered_cat"])
print("success!")


strat_train_set,strat_test_set = train_test_split(
    df_new,
    test_size=0.2,
    stratify=df["Quantity Ordered_cat"],
    random_state=42,
)

for set_ in (strat_train_set,strat_test_set):
    set_.drop("Quantity Ordered_cat",axis=1,inplace=True) # we drop the additional column after we use it to let the data clean

# Spliting the data
X_train = strat_train_set.drop("Quantity Ordered",axis=1)
y_train = strat_train_set["Quantity Ordered"]

X_test = strat_test_set.drop("Quantity Ordered",axis=1)
y_test = strat_test_set["Quantity Ordered"]

print("The Data Has been splited")

X_train.head()

      Order ID                   Product  Quantity Ordered  Price Each  \
31957   297150  Lightning Charging Cable                 1       14.95   
70790   144317          Wired Headphones                 1       11.99   
70789   144316  Lightning Charging Cable                 1       14.95   
70788   144315             Flatscreen TV                 1      300.00   
70787   144314  Lightning Charging Cable                 1       14.95   

               Order Date                       Purchase Address        Date  \
31957 2020-01-01 00:38:00        427 Wilson St, Dallas, TX 75001  2020-01-01   
70790 2019-01-18 17:08:00    335 6th St, San Francisco, CA 94016  2019-01-18   
70789 2019-01-29 16:05:00  909 Cedar St, San Francisco, CA 94016  2019-01-29   
70788 2019-01-28 18:33:00        9 Johnson St, Atlanta, GA 30301  2019-01-28   
70787 2019-01-20 11:39:00         88 Ridge St, Seattle, WA 98101  2019-01-20   

       Month  
31957      1  
70790      1  
70789      1  
70788      1  

Please refer to https://modin.readthedocs.io/en/stable/supported_apis/defaulting_to_pandas.html for explanation.


success!


# **Applying the techniques** ∮

In [ ]:
# First we need to Split the data
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,FunctionTransformer,StandardScaler
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV,HalvingGridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

# Because We have a heavy tails in the column "Price Each" we will use log10p
# First we round the decimals e.g : 15.99 -> 16
X_train["Price Each"] = X_train["Price Each"].round()
X_test["Price Each"] = X_test["Price Each"].round()
# The logarithmic Transformation because it's the most simple and effective one here
log_function = FunctionTransformer(np.log10,validate=True)
ordinal_col = ["Product"]
log_col = ["Price Each"]

ord_pipeline = Pipeline([
    ("odinal",OrdinalEncoder(
        categories=choices, # The Ranking
        handle_unknown="use_encoded_value",
        unknown_value=-1, # make any category the model didn't see in the training become -1 instead of dropping error
    )),
    ("scaler",StandardScaler())
])
log_pipeline = Pipeline([
    ("log_transformer",log_function),
    ("scaler",StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ("ord",ord_pipeline,ordinal_col),
    ("log",log_pipeline,log_col),
])
full_pipeline = Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",SVR(kernel="rbf",C=1.0,epsilon=0.1))
])

params_distrubtion = {
    "regressor__C" : np.logspace(-2,3,100),
    "regressor__gamma": np.logspace(-4,1,100),
    "regressor__epsilon": [0.01,0.05,0.1,0.2,0.5],
}

halving_random = HalvingRandomSearchCV(
    estimator=full_pipeline,
    param_distributions=params_distrubtion,
    factor=3,
    resource='n_samples',
    cv=5,
    random_state=42,
    n_jobs=-1,
)
halving_random.fit(X_train._to_pandas(),y_train._to_pandas())

best_c = halving_random.best_params_["regressor__C"]
best_gamma = halving_random.best_params_["regressor__gamma"]
best_eps = halving_random.best_params_["regressor__epsilon"]

param_grid_fine = {
    'regressor__C': [best_c * 0.5, best_c, best_c * 1.5],
    'regressor__gamma': [best_gamma * 0.5, best_gamma, best_gamma * 1.5],
    'regressor__epsilon': [best_eps * 0.8, best_eps, best_eps * 1.2],
}

halving_grid = HalvingGridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid_fine,
    factor=3,
    resource='n_samples',
    cv=5,
    n_jobs=-1,
)

halving_grid.fit(X_train._to_pandas(),y_train._to_pandas())
best_model = halving_grid.best_estimator_
print(f"best model is :  {best_model}")
